# CS224N L01 代码胶囊：负采样损失 vs 全 Softmax

> **这段代码在看什么**：对比 Word2Vec 训练中的两种目标函数——全 softmax（遍历整个词汇表）和负采样（只采样 k 个负例）。通过实际计算和训练模拟，展示负采样如何把 O(|V|) 的计算降到 O(k)，以及梯度如何「拉近正样本、推远负样本」。

> **和课堂内容的对应**：本胶囊对应 **WP05**（负采样目标函数 vs 全 softmax），锚定 Notes §3.5 Eq.14-15 和 R02 论文 Section 3。

> **运行后先看哪里**：
> 1. Part A 的概率表——全 softmax 给每个词分配的概率几乎一样（~10%），说明随机初始化时模型没有区分能力
> 2. Part B 的 sigmoid 值——所有负样本的 σ(-u^T v) 都接近 0.5
> 3. Part E 的训练曲线——正样本 cosine 上升（拉近），负样本 cosine 下降（推远）

In [ ]:
import math, random, copy

random.seed(42)
VOCAB = ["bank","river","money","loan","stock","market","cat","dog","car","book"]
VS = len(VOCAB); D = 4
w2i = {w:i for i,w in enumerate(VOCAB)}
i2w = {i:w for w,i in w2i.items()}

def mkvec(vs, d, s=42):
    random.seed(s)
    return [[random.gauss(0,0.1) for _ in range(d)] for _ in range(vs)]

V = mkvec(VS, D, 42)  # center word 向量矩阵
U = mkvec(VS, D, 99)  # outside word 向量矩阵

def dot(a,b): return sum(x*y for x,y in zip(a,b))
def sig(x):
    # sigmoid: sigma(x) = 1/(1+exp(-x))
    if x>=0: return 1.0/(1.0+math.exp(-x))
    e=math.exp(x); return e/(1.0+e)
def sfx(scores):
    # softmax: 把分数归一化成概率分布
    m=max(scores); e=[math.exp(s-m) for s in scores]; t=sum(e); return [x/t for x in e]
def vsub(a,b): return [x-y for x,y in zip(a,b)]
def vadd(a,b): return [x+y for x,y in zip(a,b)]
def vscl(a,c): return [x*c for x in a]
def vnorm(a): return math.sqrt(sum(x*x for x in a))
def cosim(a,b):
    na,nb=vnorm(a),vnorm(b)
    return dot(a,b)/(na*nb) if na and nb else 0.0

cw="bank"; pw="river"; ci=w2i[cw]; pi=w2i[pw]
K=5  # 负样本数量
negs=random.sample([i for i in range(VS) if i!=pi], K)
nw=[i2w[i] for i in negs]
print(f"词汇表 |V|={VS}, 维度 d={D}")
print(f"中心词: {cw}, 正样本: {pw}, 负样本 (k={K}): {nw}")

## Part A：全 Softmax 损失

> **这段代码在做什么**：计算中心词 "bank" 与所有 outside 词的点积，然后通过 softmax 得到概率分布。损失 = -log P(positive|center)。
>
> **先看哪里**：所有概率都接近 0.1（= 1/|V|），因为随机初始化时点积都接近 0。
>
> **输出怎么解释**：损失 = -log(0.100835) = 2.294274——随机初始化时模型对 10 个词一视同仁。

In [ ]:
vc = V[ci]
scores = [dot(vc, U[j]) for j in range(VS)]
probs = sfx(scores)
L_sm = -math.log(probs[pi])

print(f"{'Word':<10}{'Score':>12}{'P(w|c)':>12}  Label")
print("-"*50)
for j in range(VS):
    lb = ""
    if j == pi: lb = "<- positive"
    elif i2w[j] in nw: lb = "<- negative"
    print(f"{i2w[j]:<10}{scores[j]:>12.6f}{probs[j]:>12.6f}  {lb}")
print(f"\nFull Softmax loss = -log({probs[pi]:.6f}) = {L_sm:.6f}")
print(f"计算量: O(|V|) = O({VS})")

## Part B：负采样损失 (SGNS)

> **这段代码在做什么**：对同一个 (bank, river) 词对，用负采样目标计算损失。
>
> **先看哪里**：所有 sigmoid 值都接近 0.5——随机初始化时模型无法区分正负。
>
> **容易误解的地方**：负采样损失值（4.152770）比全 softmax（2.294274）大——不是 bug！两个损失函数定义不同，不能直接比较绝对值。

In [ ]:
sp = dot(vc, U[pi]); Lp = -math.log(sig(sp))
nscores = [dot(vc, U[k]) for k in negs]
Lnt = [-math.log(sig(-s)) for s in nscores]
Ln = sum(Lnt); Lsgns = Lp + Ln

print(f"Positive: -log sigma({sp:.6f}) = {Lp:.6f}")
print(f"{'Neg':<10}{'u^Tv':>10}{'sigma(-u^Tv)':>14}{'-log sig':>12}")
print("-"*48)
for ki,s in zip(negs, nscores):
    print(f"{i2w[ki]:<10}{s:>10.6f}{sig(-s):>14.6f}{-math.log(sig(-s)):>12.6f}")
print(f"{'Sum':<10}{'':>10}{'':>14}{Ln:>12.6f}")
print(f"\nSGNS loss = {Lp:.6f} + {Ln:.6f} = {Lsgns:.6f}")
print(f"计算量: O(k) = O({K+1})")

## Part C：计算量对比

> **输出怎么解释**：|V|=10, k=5 时加速比只有 2x。但当 |V|=100,000, k=5 时，加速比达到 **20,000x**。

In [ ]:
print(f"{'Method':<20}{'Loss':>12}{'Terms':>8}  Complexity")
print("-"*55)
print(f"{'Full Softmax':<20}{L_sm:>12.6f}{VS:>8}  O(|V|)=O({VS})")
print(f"{'Neg. Sampling':<20}{Lsgns:>12.6f}{K+1:>8}  O(k)=O({K+1})")
print(f"\n加速比: |V|/k = {VS/K:.1f}x; 当 |V|=100k: 20,000x")

## Part D：梯度方向分析

> **这段代码在看什么**：对比全 softmax 和负采样的梯度方向。
> - 全 softmax 梯度涉及**所有** |V| 个 outside 向量
> - 负采样梯度只涉及 **k+1** 个向量

In [ ]:
eu = [0.0]*D
for j in range(VS):
    for d in range(D): eu[d] += probs[j]*U[j][d]
g_sm = vsub(eu, U[pi])
sp_sig = sig(sp)
g_sg = vscl(U[pi], sp_sig-1.0)
for i in range(K):
    g_sg = vadd(g_sg, vscl(U[negs[i]], sig(-nscores[i])))

print(f"Full Softmax grad: [{', '.join(f'{g:.6f}' for g in g_sm)}]  norm={vnorm(g_sm):.6f}")
print(f"  touches {VS} outside vectors (ALL)")
print(f"SGNS grad: [{', '.join(f'{g:.6f}' for g in g_sg)}]  norm={vnorm(g_sg):.6f}")
print(f"  touches {K+1} outside vectors")

## Part E：训练模拟——正对拉近、负对推远

> **先看哪里**：
> - 正样本 cosine：0.7617 → 0.9438（**上升** = 拉近）
> - 负样本 avg cosine：-0.0221 → -0.6993（**下降** = 推远）
>
> **和课堂内容的对应**：Notes §3.4 的「observed minus expected」梯度直觉。

In [ ]:
Vt = copy.deepcopy(V); Ut = copy.deepcopy(U)
lr = 0.05; N = 100
hist = {"step":[], "pos":[], "neg":[]}
for step in range(N+1):
    pc = cosim(Vt[ci], Ut[pi])
    nc = [cosim(Vt[ci], Ut[k]) for k in negs]
    an = sum(nc)/len(nc)
    if step % 20 == 0 or step == N:
        hist["step"].append(step); hist["pos"].append(round(pc,4)); hist["neg"].append(round(an,4))
    if step == N: break
    vcs = list(Vt[ci])
    sp_d = dot(vcs, Ut[pi]); sg_p = sig(sp_d)
    Ut[pi] = vsub(Ut[pi], vscl(vscl(vcs, sg_p-1.0), lr))
    for ki in negs:
        sk = dot(vcs, Ut[ki]); sg_k = sig(sk)
        Ut[ki] = vsub(Ut[ki], vscl(vscl(vcs, 1.0-sg_k), lr))

fpc = cosim(Vt[ci], Ut[pi])
fan = sum(cosim(Vt[ci], Ut[k]) for k in negs)/K
print(f"{'Step':<8}{'cos(pos)':<16}{'avg cos(neg)'}")
print("-"*40)
for i in range(len(hist["step"])):
    print(f"{hist['step'][i]:<8}{hist['pos'][i]:<16.4f}{hist['neg'][i]:.4f}")
print(f"\nPositive cos: {hist['pos'][0]:.4f} -> {fpc:.4f} (closer)")
print(f"Negative avg cos: {hist['neg'][0]:.4f} -> {fan:.4f} (apart)")

## 总结

> **核心结论**：负采样用 k 个二分类替代 |V| 分类，保持「拉近正样本、推远负样本」的梯度方向，计算量从 O(|V|) 降到 O(k)。当 |V|=100,000 时，这是 20,000 倍加速。